In [2]:
# === Setup: paths + helper ===
from pathlib import Path
import time, pandas as pd
from project_package.modeling import (
    train_classification_from_csv,
    train_regression_from_csv,
)

ROOT = Path.cwd()
OUT  = ROOT / "supervised"
OUT.mkdir(parents=True, exist_ok=True)

CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / CSV.name
    if alt.exists(): CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

IDS = ["Booking ID", "Customer ID"]  # keep in CSV exports

def timed(fn, **kw):
    t0 = time.time()
    res = fn(**kw)
    mins = (time.time() - t0) / 60
    return res, mins

def save_report(report: dict, path: Path):
    pd.DataFrame([report]).to_csv(path, index=False)


In [3]:
# === Problem #1: Classification (completion vs cancellation) ===
cls_res, t_cls = timed(
    train_classification_from_csv,
    csv_path=str(CSV),
    target_col=None,            # derive _target_completed_ from "Booking Status"
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report(cls_res.report, OUT / f"cls_best_{cls_res.best_model_name}_report.csv")

print("=== Completion (classification) ===")
print("time (min):", round(t_cls, 2))
print("best model:", cls_res.best_model_name)
print("report   :", cls_res.report)
print("preds CSV:", cls_res.preds_csv_path)
print("split CSV:", cls_res.split_csv_path)
print("model PKL:", cls_res.model_path)


c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


=== Completion (classification) ===
time (min): 25.04
best model: rf
report   : {'model': 'rf', 'accuracy': 0.9374893428366811, 'precision': 0.9126375290198849, 'recall': 0.9943909815782238, 'f1': 0.9517618884707494, 'roc_auc': 0.9684372369868492}
preds CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\cls_best_rf_preds.csv
split CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_classification.csv
model PKL: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_cls_rf__target_completed_.pkl


In [4]:
# === Problem #2: Regression (fare) ===
fare_target = "Booking Value_fill_scaled"  
fare_res, t_fare = timed(
    train_regression_from_csv,
    csv_path=str(CSV),
    target_col=fare_target,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report(fare_res.report, OUT / f"reg_best_{fare_target.replace(' ','_')}_{fare_res.best_model_name}_report.csv")

print("=== Fare (regression) ===")
print("time (min):", round(t_fare, 2))
print("best model:", fare_res.best_model_name)
print("report   :", fare_res.report)
print("preds CSV:", fare_res.preds_csv_path)
print("split CSV:", fare_res.split_csv_path)
print("model PKL:", fare_res.model_path)


=== Fare (regression) ===
time (min): 18.42
best model: tree
report   : {'model': 'tree', 'MAE': 0.986091067841411, 'RMSE': 1.6794663952263587, 'R2': -0.0005764767983282848}
preds CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\reg_best_Booking_Value_fill_scaled_tree_preds.csv
split CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_regression_Booking_Value_fill_scaled.csv
model PKL: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_reg_tree_Booking Value_fill_scaled.pkl


In [5]:
# === Problem #3: Regression (customer rating) ===
rating_target = "Customer Rating_fill"
rating_res, t_rating = timed(
    train_regression_from_csv,
    csv_path=str(CSV),
    target_col=rating_target,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report(rating_res.report, OUT / f"reg_best_{rating_target.replace(' ','_')}_{rating_res.best_model_name}_report.csv")

print("=== Rating (regression) ===")
print("time (min):", round(t_rating, 2))
print("best model:", rating_res.best_model_name)
print("report   :", rating_res.report)
print("preds CSV:", rating_res.preds_csv_path)
print("split CSV:", rating_res.split_csv_path)
print("model PKL:", rating_res.model_path)


=== Rating (regression) ===
time (min): 18.67
best model: tree
report   : {'model': 'tree', 'MAE': 0.216340046624083, 'RMSE': 0.3469420037365729, 'R2': -0.0005331047369210307}
preds CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\reg_best_Customer_Rating_fill_tree_preds.csv
split CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_regression_Customer_Rating_fill.csv
model PKL: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_reg_tree_Customer Rating_fill.pkl


The below is the lite version.  Choose the 15-25 best features to model and check the performance.
Also, not using all the records((15-30%) of total data)) to reduce computation time.

In [6]:
# === LITE: Setup & Helpers (fixed) ===
# Purpose:
# - Resolve CSV path
# - Robust target resolution (works even if "Booking Status" is missing)
# - Build reduced CSVs (top-K numeric by |corr| + optional stratified sampling)
# - Build (X, y, preprocessor) for sensitivity/learning-curve
# - Timing & model-size printer

from pathlib import Path
import os, time, numpy as np, pandas as pd

# Import from your project package (uses your modeling.py)
from project_package.modeling import (
    EXCLUDE_ALWAYS, load_csv_dedup, make_binary_target,
    pre_ohe_scaled, pre_ordinal,
    train_classification_from_csv, train_regression_from_csv
)

# ---------- Paths ----------
ROOT = Path.cwd()
CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / CSV.name
    if alt.exists():
        CSV = alt
assert CSV.exists(), f"[LITE] CSV not found: {CSV}"

# LITE outputs (kept separate from full runs)
OUT = ROOT / "supervised_lite"
OUT.mkdir(parents=True, exist_ok=True)

# ID columns to carry in CSV exports (not used as features)
IDS = ["Booking ID", "Customer ID"]

# LITE knobs (speed vs fidelity)
K_FEATURES   = 20       # keep top-K numeric features by |corr|
SAMPLE_FRAC  = 0.30     # use 15–30% rows; stratified if target is binary
RANDOM_STATE = 42

# ---------- Small utilities ----------
def run_and_print(train_fn, **kw):
    """
    Run a training function and print:
      - elapsed minutes
      - model pickle size (MB)
      - artifact paths
    """
    t0 = time.time()
    res = train_fn(**kw)
    mins = (time.time() - t0) / 60
    try:
        size_mb = os.path.getsize(res.model_path) / (1024 * 1024)
    except Exception:
        size_mb = float("nan")
    print(f"  time (min): {mins:.2f}")
    print(f"  model size: {size_mb:.1f} MB")
    print(f"  preds CSV : {res.preds_csv_path}")
    print(f"  split CSV : {res.split_csv_path}")
    print(f"  model PKL : {res.model_path}")
    return res

def _to_series(x) -> pd.Series:
    """Force any pandas object into a clean 1-D Series with SAME row count."""
    if isinstance(x, pd.Series):
        return x
    if isinstance(x, pd.DataFrame):
        return x.iloc[:, 0]
    arr = np.asarray(x)
    if arr.ndim > 1:
        arr = arr[:, 0]
    return pd.Series(arr)

def _is_binary(y_like) -> bool:
    """Check if y is binary {0,1} after numeric coercion."""
    y = _to_series(y_like)
    y_num = pd.to_numeric(y, errors="coerce").to_numpy()
    uniq = np.unique(y_num[~np.isnan(y_num)])
    return set(np.unique(uniq.astype(int))) <= {0, 1}

def resolve_target(df: pd.DataFrame, target_col: str | None) -> tuple[pd.DataFrame, str]:
    """
    Resolve/ensure a target column:
      - If target_col is provided and exists -> use it
      - Else if 'Booking Status' exists -> derive _target_completed_ (0/1)
      - Else if '_target_completed_' exists -> use it
      - Else raise with helpful column sample
    """
    if target_col and target_col in df.columns:
        return df, target_col
    if "Booking Status" in df.columns:
        df2, tgt = make_binary_target(df)
        return df2, tgt
    if "_target_completed_" in df.columns:
        return df, "_target_completed_"
    # build a safe error string (no literal { } inside f-strings)
    wanted = ["Booking Status", "_target_completed_"] + ([target_col] if target_col else [])
    raise KeyError(f"Cannot resolve target; none of these columns exist: {wanted}")

def topk_numeric_corr_view(csv_path: str | Path,
                           target_col: str | None,
                           k: int,
                           sample_frac: float,
                           ids: list[str] | None = None,
                           random_state: int = 42) -> tuple[Path, str]:
    """
    Build a reduced CSV for fast experiments:
      1) Ensure/derive target (if None -> _target_completed_).
      2) Rank numeric features by |corr| w/ target; keep top-k.
      3) Keep [ids + target + top-k numeric].
      4) Row sampling (stratified if target is binary {0,1}).
    Returns: (reduced_csv_path, resolved_target_col)
    """
    ids = ids or ["Booking ID", "Customer ID"]
    csv_path = Path(csv_path)
    df = load_csv_dedup(str(csv_path))
    df, tgt = resolve_target(df, target_col)

    # Candidate features = all minus leakage, ids, target
    drop_cols = set(EXCLUDE_ALWAYS) | set(ids) | {tgt}
    X = df.drop(columns=[c for c in df.columns if c in drop_cols], errors="ignore")

    # Rank numeric by |corr| with target
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    if not num_cols:
        raise ValueError("No numeric columns after dropping leakage/IDs/target.")
    corr_vec = df[num_cols + [tgt]].corr(numeric_only=True)[tgt]
    corr_vec = corr_vec.abs().dropna().sort_values(ascending=False)
    keep_num = corr_vec.head(int(k)).index.tolist()

    # Build reduced dataframe (IDs + target + top-K numeric)
    cols = [c for c in (ids + [tgt] + keep_num) if c in df.columns]
    df_small = df[cols].copy()

    # Row sampling (stratified if binary)
    if sample_frac < 1.0:
        y = _to_series(df_small[tgt])
        if _is_binary(y):
            tmp = "_tmp_strat_y_"
            df_small[tmp] = pd.to_numeric(y, errors="coerce").fillna(-1).astype(int).values
            df_small = (
                df_small
                .groupby(tmp, group_keys=False)
                .apply(lambda g: g.sample(frac=sample_frac, random_state=random_state))
                .reset_index(drop=True)
            )
            df_small.drop(columns=[tmp], inplace=True)
        else:
            df_small = df_small.sample(frac=sample_frac, random_state=random_state).reset_index(drop=True)

    out = csv_path.with_name(csv_path.stem + f"__top{int(k)}_s{int(sample_frac*100)}.csv")
    df_small.to_csv(out, index=False)
    print(f"[LITE] reduced CSV -> {out.name} (rows={len(df_small)}, cols={len(df_small.columns)}) | target={tgt}")
    return out, tgt

def build_xy_for_problem(csv_path: str | Path,
                         target_col: str | None,
                         id_cols: list[str],
                         use_topk: int = 20,
                         sample_frac: float = 0.30,
                         for_trees: bool = False,
                         random_state: int = 42):
    """
    Produce (X_df, y_series, preprocessor, resolved_target_col, reduced_csv_path)
    using the same reduction strategy as the LITE training:
      - top-K numeric by |corr| with target
      - optional stratified sampling
      - tree vs non-tree preprocessing choice
    """
    reduced_csv, tgt = topk_numeric_corr_view(
        csv_path, target_col=target_col, k=use_topk,
        sample_frac=sample_frac, ids=id_cols, random_state=random_state
    )
    df_small = load_csv_dedup(str(reduced_csv))
    y = _to_series(df_small[tgt])
    X = df_small.drop(columns=[tgt] + [c for c in id_cols if c in df_small.columns], errors="ignore")
    pre = pre_ordinal(X) if for_trees else pre_ohe_scaled(X)
    return X, y, pre, tgt, reduced_csv


In [7]:
# === LITE: Problem #1 — Classification (completion vs cancellation) ===
# Build reduced CSV for classification target and train quickly.
csv_lite_cls, cls_tgt = topk_numeric_corr_view(
    CSV, target_col=None, k=K_FEATURES, sample_frac=SAMPLE_FRAC, ids=IDS, random_state=RANDOM_STATE
)
print("=== LITE: Completion (classification) ===")
_ = run_and_print(
    train_classification_from_csv,
    csv_path=str(csv_lite_cls),
    target_col=cls_tgt,               # explicitly pass derived target if needed
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=RANDOM_STATE,
    tune_row_cap=None,                # already lite
)


[LITE] reduced CSV -> ncr_ride_bookings_with_weather_filled_scaled_short__top20_s30.csv (rows=43984, cols=23) | target=_target_completed_
=== LITE: Completion (classification) ===
  time (min): 0.25
  model size: 0.0 MB
  preds CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\cls_best_logreg_preds.csv
  split CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\split_assignments_classification.csv
  model PKL : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\best_cls_logreg__target_completed_.pkl


In [8]:
# === LITE: Problem #2 — Regression (fare) ===
fare_tgt = "Booking Value_fill_scaled"
csv_lite_fare, _ = topk_numeric_corr_view(
    CSV, target_col=fare_tgt, k=K_FEATURES, sample_frac=SAMPLE_FRAC, ids=IDS, random_state=RANDOM_STATE
)
print("=== LITE: Fare (regression) ===")
_ = run_and_print(
    train_regression_from_csv,
    csv_path=str(csv_lite_fare),
    target_col=fare_tgt,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=RANDOM_STATE,
    tune_row_cap=None,
)


[LITE] reduced CSV -> ncr_ride_bookings_with_weather_filled_scaled_short__top20_s30.csv (rows=43984, cols=23) | target=Booking Value_fill_scaled
=== LITE: Fare (regression) ===
  time (min): 0.15
  model size: 0.0 MB
  preds CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\reg_best_Booking_Value_fill_scaled_ridge_preds.csv
  split CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\split_assignments_regression_Booking_Value_fill_scaled.csv
  model PKL : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\best_reg_ridge_Booking Value_fill_scaled.pkl


In [9]:
# === LITE: Problem #3 — Regression (customer rating) ===
rating_tgt = "Customer Rating_fill"
csv_lite_rating, _ = topk_numeric_corr_view(
    CSV, target_col=rating_tgt, k=K_FEATURES, sample_frac=SAMPLE_FRAC, ids=IDS, random_state=RANDOM_STATE
)
print("=== LITE: Rating (regression) ===")
_ = run_and_print(
    train_regression_from_csv,
    csv_path=str(csv_lite_rating),
    target_col=rating_tgt,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=RANDOM_STATE,
    tune_row_cap=None,
)


[LITE] reduced CSV -> ncr_ride_bookings_with_weather_filled_scaled_short__top20_s30.csv (rows=43984, cols=23) | target=Customer Rating_fill
=== LITE: Rating (regression) ===
  time (min): 0.15
  model size: 0.0 MB
  preds CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\reg_best_Customer_Rating_fill_tree_preds.csv
  split CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\split_assignments_regression_Customer_Rating_fill.csv
  model PKL : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\best_reg_tree_Customer Rating_fill.pkl


In [10]:
from project_package.modeling import load_csv_dedup, make_binary_target, EXCLUDE_ALWAYS
import pandas as pd
from pathlib import Path

CSV = Path("ncr_ride_bookings_with_weather_filled_scaled_short.csv")
IDS = ["Booking ID", "Customer ID"]

def show_topk(csv_path, target_col, k=20, ids=IDS):
    df = load_csv_dedup(str(csv_path))
    if target_col is None or target_col not in df.columns:
        df, target_col = make_binary_target(df)  # derives _target_completed_
    drop = set(EXCLUDE_ALWAYS) | set(ids) | {target_col}
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    num = X.select_dtypes(include=["number"])
    corr = pd.concat([num, df[[target_col]]], axis=1).corr(numeric_only=True)[target_col]
    topk = corr.abs().dropna().sort_values(ascending=False).head(k)
    print(f"Target = {target_col} | top-{k} numeric features by |corr|:")
    display(topk)

# Problem 1: completion (classification)
show_topk(CSV, target_col=None, k=20)  # uses derived _target_completed_

# Problem 2: fare (regression)
show_topk(CSV, target_col="Booking Value_fill_scaled", k=20)

# Problem 3: customer rating (regression)
show_topk(CSV, target_col="Customer Rating_fill", k=20)


Target = _target_completed_ | top-20 numeric features by |corr|:


_target_completed_                       1.000000
pick_latitude                            0.199323
Ride Distance_fill_scaled                0.151109
pick_longitude                           0.149788
drop_latitude                            0.058931
drop_longitude                           0.049321
pick_station_longitude                   0.006475
snowfall_dropoff_log_scaled              0.003162
precipitation_dropoff_log_scaled         0.002846
rain_dropoff_log_scaled                  0.002728
wind_speed_10m_scaled                    0.002614
wind_speed_10m_scaled.1                  0.002614
wind_speed_10m_dropoff_scaled.1          0.002413
wind_speed_10m_dropoff_scaled            0.002413
rain_log_scaled.1                        0.002381
rain_log_scaled                          0.002381
pick_station_latitude                    0.002366
precipitation_log_scaled.1               0.002313
precipitation_log_scaled                 0.002313
relative_humidity_2m_dropoff_scaled.1    0.001947


Target = Booking Value_fill_scaled | top-20 numeric features by |corr|:


Booking Value_fill_scaled                1.000000
apparent_temperature_dropoff_scaled      0.004953
apparent_temperature_dropoff_scaled.1    0.004953
Ride Distance_fill_scaled                0.004161
dew_point_2m_scaled                      0.004051
dew_point_2m_scaled.1                    0.004051
temperature_2m_dropoff_scaled            0.003974
temperature_2m_dropoff_scaled.1          0.003974
precipitation_dropoff_log_scaled         0.003834
rain_dropoff_log_scaled                  0.003756
dew_point_2m_dropoff_scaled.1            0.003714
dew_point_2m_dropoff_scaled              0.003714
apparent_temperature_scaled.1            0.003644
apparent_temperature_scaled              0.003644
pick_station_latitude                    0.002857
wind_speed_10m_scaled                    0.002854
wind_speed_10m_scaled.1                  0.002854
drop_station_longitude                   0.002825
temperature_2m_scaled.1                  0.002483
temperature_2m_scaled                    0.002483


Target = Customer Rating_fill | top-20 numeric features by |corr|:


Customer Rating_fill                     1.000000
drop_longitude                           0.005253
pick_longitude                           0.004803
Ride Distance_fill_scaled                0.004213
drop_station_latitude                    0.003299
temperature_2m_dropoff_scaled            0.003124
temperature_2m_dropoff_scaled.1          0.003124
apparent_temperature_dropoff_scaled.1    0.002783
apparent_temperature_dropoff_scaled      0.002783
pick_station_latitude                    0.002406
relative_humidity_2m_scaled              0.002193
relative_humidity_2m_scaled.1            0.002193
pick_latitude                            0.002091
temperature_2m_scaled                    0.001822
temperature_2m_scaled.1                  0.001822
apparent_temperature_scaled              0.001795
apparent_temperature_scaled.1            0.001795
relative_humidity_2m_dropoff_scaled.1    0.001759
relative_humidity_2m_dropoff_scaled      0.001759
snowfall_dropoff_log_scaled              0.001755
